# Clase 011 — pathlib

**Parte 0** · `pathlib` docs + PEP 428.

> 🎯 API moderna y multiplataforma para todo lo de filesystem. Adiós a `os.path.join`.

> ⏱️ ~60 min

## ⚙️ Setup

In [ ]:
import tempfile
from pathlib import Path
import time

# Carpeta de trabajo temporal
base = Path(tempfile.mkdtemp(prefix='lab011_'))
print(f'trabajo en: {base}')

## 1️⃣ `Path` vs string

```python
# ❌ Vieja escuela
import os
path = os.path.join(os.path.expanduser('~'), 'datos', '2026', 'enero.csv')

# ✅ pathlib
from pathlib import Path
path = Path.home() / 'datos' / '2026' / 'enero.csv'
```

El operador `/` se sobreescribe en `Path` para componer rutas. **Es multiplataforma**: en Windows se renderiza con `\`, en Unix con `/`.

In [ ]:
p = Path.home() / 'datos' / '2026' / 'enero.csv'
print(f'path: {p}')
print(f'parent: {p.parent}')
print(f'name: {p.name}')
print(f'stem: {p.stem}')
print(f'suffix: {p.suffix}')
print(f'parts: {p.parts}')

## 2️⃣ Crear, leer, escribir

One-liners cubren el 90% de los casos:

In [ ]:
# Crear estructura
(base / 'subdir').mkdir(parents=True, exist_ok=True)

# Escribir texto
(base / 'hola.txt').write_text('hola mundo\nsegunda línea\n', encoding='utf-8')

# Leer texto
contenido = (base / 'hola.txt').read_text(encoding='utf-8')
print('contenido:')
print(contenido)

# Binario
(base / 'datos.bin').write_bytes(b'\x00\x01\x02\x03')
print('bytes:', (base / 'datos.bin').read_bytes())

## 3️⃣ Listar archivos

- `path.iterdir()` — listado simple (no recursivo)
- `path.glob('*.csv')` — patrón en un nivel
- `path.rglob('*.py')` — recursivo (todo el árbol)

In [ ]:
# Genera archivos de muestra
for ext in ['csv', 'csv', 'txt', 'py', 'csv']:
    nombre = f'archivo_{ext}_{int(time.time()*1000)%10000}.{ext}'
    (base / nombre).write_text(f'demo {ext}')

# Solo CSVs, ordenados por tamaño
csvs = sorted(base.glob('*.csv'), key=lambda p: p.stat().st_size, reverse=True)
for p in csvs:
    print(f'  {p.name:35s} {p.stat().st_size} bytes')

## 4️⃣ Operaciones útiles

```python
p.exists()         # bool
p.is_file()        # bool
p.is_dir()         # bool
p.absolute()       # ruta absoluta (sin resolver symlinks)
p.resolve()        # ruta absoluta + resuelve symlinks
p.unlink()         # borra archivo
p.rmdir()          # borra dir vacío
p.rename(nuevo)    # renombra/mueve
p.stat().st_size   # info filesystem (tamaño, mtime, etc.)
p.with_suffix('.json')   # cambia extensión
```

In [ ]:
# Demostración
p = base / 'hola.txt'
print(f'absolute      : {p.absolute()}')
print(f'with_suffix   : {p.with_suffix(".md")}')
print(f'with_name     : {p.with_name("otro.txt")}')
print(f'mtime         : {p.stat().st_mtime:.0f}')
print(f'size          : {p.stat().st_size} bytes')

## 5️⃣ Rutas relativas al script — `__file__`

**Problema clásico**: tu script carga `data.csv` con `pd.read_csv('data.csv')` y funciona desde el directorio del proyecto, pero falla cuando lo ejecutan desde otro lado.

**Solución**: rutas relativas al script, no al cwd:

```python
from pathlib import Path

ROOT = Path(__file__).parent
df = pd.read_csv(ROOT / 'data' / 'penguins.csv')
```

`__file__` apunta al archivo Python actual. `.parent` da su carpeta. `.resolve()` lo convierte en absoluto.

## ✅ Checklist

- [ ] Uso `Path(...) / 'sub' / 'file'` en vez de strings
- [ ] Conozco `read_text` / `write_text` / `read_bytes`
- [ ] Uso `glob` y `rglob` según el alcance
- [ ] `mkdir(parents=True, exist_ok=True)` es mi default
- [ ] Rutas relativas a `__file__`, no al cwd

## 📝 Homework

Ver `README.md`. Script `inventario.py` que recorre un directorio y produce CSV con metadata.

## 📖 Definiciones y características

**`Path`**

Objeto que representa una ruta de filesystem orientada a objetos (`pathlib.Path`). Sobreescribe `/` para componer rutas, multiplataforma (Windows usa `\`, Unix `/`, transparente). Tiene métodos para casi todo: leer, escribir, listar, mover, borrar.

**Ruta absoluta vs relativa**

**Absoluta**: empieza desde la raíz (`C:\dev\proyecto\data.csv` o `/home/user/data.csv`). **Relativa**: parte del cwd (`data.csv`). Las rutas relativas dependen de dónde se ejecuta — fuente de bugs.

**`Path.cwd()` vs `Path(__file__).parent`**

`cwd()` es el directorio donde se ejecutó el script (cambia según el invocante). `__file__` es la ruta al archivo Python actual; `.parent` su carpeta. **Usa `__file__`** para recursos que viven junto al script.

**`glob` vs `rglob`**

Patrones tipo shell. `glob('*.csv')` busca en el directorio actual (1 nivel). `rglob('*.csv')` busca recursivo (todo el árbol). `**` significa 'cualquier número de directorios'.

**`read_text` / `write_text`**

One-liners para texto: `Path('x.txt').write_text(contenido, encoding='utf-8')`. Para binario: `read_bytes` / `write_bytes`. Para JSON/CSV usa librerías especializadas.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `FileNotFoundError` aunque el archivo existe | Estás usando ruta relativa y el cwd no es el que crees. **Fix**: `print(Path.cwd())` para diagnosticar; usa rutas relativas a `__file__` para archivos del proyecto. |
| `PermissionError` al escribir | Carpeta read-only, OneDrive sincronizando, antivirus bloqueando. **Fix**: chequea permisos con `p.stat()`, escribe a `tempfile.gettempdir()` para tests. |
| `mkdir()` falla si el directorio ya existe | Default es `exist_ok=False`. **Fix**: `p.mkdir(parents=True, exist_ok=True)` — crea árbol completo idempotente. |
| Mezclo `os.path` y `pathlib` | Algunos funciones esperan strings (`open()`, `pd.read_csv()`). Path soporta el protocolo `os.PathLike` y la mayoría las acepta directo; si no, `str(p)` lo convierte. |
| `glob('*.CSV')` no encuentra `archivo.csv` | Case-sensitive en Linux, insensible en Windows. **Fix**: filtra explícito con `[p for p in path.iterdir() if p.suffix.lower() == '.csv']`. |

## ❓ Preguntas frecuentes

**❓ ¿`pathlib` o `os.path`?**

**`pathlib`** para código nuevo. `os.path` es la API funcional vieja (strings + funciones); pathlib es OO y mucho más legible. Solo usa `os.path` para compatibilidad con código viejo.

**❓ ¿Cómo evito el típico `'C:/Users/...'` vs `'/home/...'` cross-platform?**

**No hardcodees rutas absolutas.** Usa `Path.home()`, `Path(__file__).parent`, `tempfile.gettempdir()`. Y siempre `Path` + `/`, nunca strings concatenados.

**❓ ¿Cómo leo un CSV grande con pathlib?**

Pathlib es para *paths*, no parsing. Combina: `pd.read_csv(Path('data') / 'big.csv')`. El Path se convierte a string automáticamente.

**❓ ¿`Path('a') / 'b/c'` o `Path('a') / 'b' / 'c'`?**

Ambas funcionan: `Path` parsea separadores. Pero la primera es menos explícita; prefiere la segunda.

**❓ ¿`shutil` o `pathlib` para mover/copiar?**

**`shutil`** para operaciones recursivas (copytree, rmtree, move) — pathlib solo cubre operaciones simples. Combinable: `shutil.copy(src_path, dst_path)` acepta Path directo.

## 🔗 Referencias

- [`pathlib` docs](https://docs.python.org/3/library/pathlib.html)
- [PEP 428](https://peps.python.org/pep-0428/)

➡️ **Siguiente:** [012 — Logging](../012-logging/README.md)

## ✅ Soluciones de los ejercicios

Intentá resolverlos vos primero; acá está una solución de referencia comentada. Todo el I/O usa `tempfile`, así que el notebook corre headless sin tocar tus carpetas.

**Ejercicio 1.** Construir una ruta multiplataforma y mostrar cómo se ve en Windows vs Unix.

In [ ]:
# Solución 1 — misma ruta lógica, distinto separador según SO
from pathlib import PureWindowsPath, PurePosixPath, Path

partes = ("datos", "2026", "enero.csv")

# PurePosix/PureWindows NO tocan el disco: solo modelan la sintaxis de rutas
ruta_unix = PurePosixPath("/home/usuario").joinpath(*partes)
ruta_win  = PureWindowsPath("C:/Users/usuario").joinpath(*partes)

print("Unix   :", ruta_unix)     # /home/usuario/datos/2026/enero.csv
print("Windows:", ruta_win)      # C:\Users\usuario\datos\2026\enero.csv

assert str(ruta_unix) == "/home/usuario/datos/2026/enero.csv"
assert str(ruta_win) == r"C:\Users\usuario\datos\2026\enero.csv"
# En código real usás Path (elige el estilo del SO actual automáticamente)
assert (Path.home() / "datos" / "2026" / "enero.csv").name == "enero.csv"
print("OK: el operador / compone rutas multiplataforma")

**Ejercicio 2.** En una carpeta con archivos mixtos (`.csv`, `.txt`, `.py`), listar solo los `.csv` ordenados por tamaño.

In [ ]:
# Solución 2 — filtrar por extensión y ordenar por tamaño
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as d:
    base = Path(d)
    # Creamos archivos con tamaños distintos
    (base / "a.csv").write_text("x" * 30, encoding="utf-8")
    (base / "b.csv").write_text("x" * 10, encoding="utf-8")
    (base / "c.csv").write_text("x" * 20, encoding="utf-8")
    (base / "notas.txt").write_text("no soy csv", encoding="utf-8")
    (base / "script.py").write_text("print(1)", encoding="utf-8")

    # suffix.lower() -> robusto ante .CSV en mayúsculas
    csvs = [f for f in base.iterdir() if f.is_file() and f.suffix.lower() == ".csv"]
    csvs_por_tamano = sorted(csvs, key=lambda f: f.stat().st_size)

    nombres = [f.name for f in csvs_por_tamano]
    tamanos = [f.stat().st_size for f in csvs_por_tamano]
    assert nombres == ["b.csv", "c.csv", "a.csv"]   # 10, 20, 30
    assert tamanos == [10, 20, 30]
    print("CSVs por tamaño:", list(zip(nombres, tamanos)))

**Ejercicio 3.** En un árbol de carpetas, encontrar todos los `.py` que contengan la palabra `TODO` en su contenido (búsqueda recursiva con `rglob`).

In [ ]:
# Solución 3 — rglob para recorrer el árbol + filtrar por contenido
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as d:
    base = Path(d)
    (base / "pkg" / "sub").mkdir(parents=True, exist_ok=True)
    (base / "app.py").write_text("# TODO: refactor\nprint(1)", encoding="utf-8")
    (base / "pkg" / "util.py").write_text("def f():\n    return 2", encoding="utf-8")
    (base / "pkg" / "sub" / "core.py").write_text("x = 1  # TODO revisar", encoding="utf-8")
    (base / "notas.txt").write_text("TODO en un txt (no cuenta)", encoding="utf-8")

    con_todo = sorted(
        f for f in base.rglob("*.py")          # ** recursivo, solo .py
        if "TODO" in f.read_text(encoding="utf-8")
    )
    relativos = [str(f.relative_to(base).as_posix()) for f in con_todo]
    assert relativos == ["app.py", "pkg/sub/core.py"]
    print("Archivos .py con TODO:", relativos)

**Ejercicio 4.** Generar 3 archivos `.txt` con `write_text`, leerlos con `read_text` y concatenarlos en uno solo.

In [ ]:
# Solución 4 — write_text / read_text y concatenación
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as d:
    base = Path(d)
    partes = {"1.txt": "uno", "2.txt": "dos", "3.txt": "tres"}
    for nombre, contenido in partes.items():
        (base / nombre).write_text(contenido + "\n", encoding="utf-8")

    # Leer en orden y concatenar
    archivos = sorted(base.glob("*.txt"))
    combinado = "".join(f.read_text(encoding="utf-8") for f in archivos)
    destino = base / "combinado.txt"
    destino.write_text(combinado, encoding="utf-8")

    assert destino.read_text(encoding="utf-8") == "uno\ndos\ntres\n"
    print("Contenido concatenado:\n" + destino.read_text(encoding="utf-8").strip())

**Ejercicio 5.** Cargar un recurso que vive *al lado del script* con `Path(__file__).parent / 'data.csv'`, no relativo al cwd. En el notebook simulamos `__file__` con un directorio temporal y cambiamos el cwd a otro lado para probar que igual encuentra el archivo.

In [ ]:
# Solución 5 — recurso relativo al script, no al cwd
import os
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as script_home, tempfile.TemporaryDirectory() as otro_cwd:
    script_dir = Path(script_home)
    # En un .py real esto sería: script_dir = Path(__file__).resolve().parent
    (script_dir / "data.csv").write_text("col\n1\n2\n", encoding="utf-8")

    # Nos paramos en OTRO directorio a propósito (simula ejecutar desde otro cwd)
    prev = Path.cwd()
    os.chdir(otro_cwd)
    try:
        # ❌ ruta relativa al cwd: NO encontraría el archivo
        assert not (Path.cwd() / "data.csv").exists()
        # ✅ ruta relativa al script: SIEMPRE lo encuentra
        data = script_dir / "data.csv"
        assert data.exists()
        contenido = data.read_text(encoding="utf-8")
        assert contenido.splitlines() == ["col", "1", "2"]
        print("OK: recurso hallado vía script_dir sin depender del cwd")
        print("cwd actual:", Path.cwd().name, "| script_dir:", script_dir.name)
    finally:
        os.chdir(prev)